In [2]:
import json
from glob import glob

import pandas as pd

flores = glob('flores/devtest/*.json')
for l in flores:
    with open(l) as f:
        language = l.split('/')[-1].split('.')[0]
        with open(f'flores/{language}.txt', 'w') as out:
            data = json.load(f)
            for d in data:
                out.write(d['text'] + '\n')
                

In [12]:
from glob import glob
from pathlib import Path
langs = glob('flores/*.txt')
utf8_langs = []
for lang in langs:
    lang_name =lang.split('/')[-1].split('.')[0]
    l = 0
    sents = Path(lang).read_text().strip().split('\n')
    for sent in sents:
        l+=len(sent.encode('utf-8'))
        # l+=len(sent)
    utf8_langs.append({'lang':lang_name, 'value':l})
    

In [13]:
pd.DataFrame(utf8_langs).to_csv('langs_utf8.csv',index=False)

In [25]:
import json
from glob import glob
import numpy as np
import math
import pandas as pd
vocab_size = [8192, 32768, 49152, 65536, 81920]
eval_data = ['de', 'de-ar','de-arp', 'de-p']
metrics = ['bpb', 'cpb', 'ppl', 'sent-nll', 'mrr', 'token-nll']

paraphrase = []
for metric in metrics:
    for vocab in vocab_size:
        de = pd.read_csv(f'ppl_results_{metric}_de/gpt2_small_DE_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')
        de_ar = pd.read_csv(f'ppl_results_{metric}_de-ar/gpt2_small_DE_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')
        de_arp = pd.read_csv(f'ppl_results_{metric}_de-arp/gpt2_small_DE_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')
        de_p = pd.read_csv(f'ppl_results_{metric}_de-p/gpt2_small_DE_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')
        lowest=[]
        highest=[]
        for d, d_ar, d_arp, d_p in zip(de, de_ar, de_arp, de_p):
            sent_d = d['Epoch-10']
            sent_d_ar = d_ar['Epoch-10']
            sent_d_arp = d_arp['Epoch-10']
            sent_d_p = d_p['Epoch-10']
            lowest.append(min([sent_d, sent_d_ar]))
            highest.append(max([sent_d, sent_d_ar]))
        if metric=='ppl':
            lowest_ave = np.exp(np.mean(np.log(lowest)))
            highest_ave = np.exp(np.mean(np.log(highest)))
        else:
            lowest_ave = sum(lowest)/len(lowest)
            highest_ave = sum(highest)/len(highest)
        paraphrase.append({'lang':'DE','tokenization':'bpe','vocab_size':vocab,'eval_data': 'DE_lowest','metric_type':metric,'mean_value':lowest_ave})
        
        paraphrase.append({'lang':'DE','tokenization':'bpe','vocab_size':vocab,'eval_data': 'DE_highest','metric_type':metric,'mean_value':highest_ave})
        
pd.DataFrame(paraphrase).to_csv('paraphrase_results_ap.csv',index=False)
        